In [2]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

In [3]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

# NLP
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import nltk
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords

print("Imports done")

Imports done


In [4]:
df = pd.read_csv('../data/processed/kairos_unified.csv')
print(f"Loaded {len(df)} records")
print(df['cascade_stage'].value_counts())

Loaded 240448 records
cascade_stage
crisis        232035
depression      7698
stress           715
Name: count, dtype: int64


In [5]:
analyzer = SentimentIntensityAnalyzer()

def get_sentiment_features(text):
    scores = analyzer.polarity_scores(str(text))
    return {
        'sentiment_neg': scores['neg'],
        'sentiment_neu': scores['neu'],
        'sentiment_pos': scores['pos'],
        'sentiment_compound': scores['compound']
    }

print("Testing on a sample:")
test = "I don't want to do anything anymore. Everything feels pointless and I'm exhausted."
print(get_sentiment_features(test))

Testing on a sample:
{'sentiment_neg': 0.253, 'sentiment_neu': 0.747, 'sentiment_pos': 0.0, 'sentiment_compound': -0.4063}


In [ ]:
# Word lists for each psychological signal
ISOLATION_WORDS = {
    'alone', 'lonely', 'isolated', 'nobody', 'no one', 'empty',
    'disconnected', 'invisible', 'forgotten', 'abandoned', 'withdrawn',
    'distant', 'detached', 'numb', 'hollow'
}

HOPELESSNESS_WORDS = {
    'hopeless', 'pointless', 'worthless', 'useless', 'meaningless',
    'nothing', 'never', 'impossible', 'stuck', 'trapped', 'forever',
    'always', 'failure', 'failed', 'cant', 'cannot', 'give up',
    'no point', 'whats the point', 'tired of'
}

BURNOUT_WORDS = {
    'exhausted', 'drained', 'burnt out', 'burned out', 'overwhelmed',
    'overworked', 'no energy', 'can\'t keep up', 'falling behind',
    'too much', 'breaking point', 'can\'t cope', 'running on empty',
    'no motivation', 'going through motions'
}

SELF_BLAME_WORDS = {
    'my fault', 'blame myself', 'i failed', 'i ruined', 'because of me',
    'i should have', 'i could have', 'i\'m the problem', 'hate myself',
    'stupid', 'idiot', 'pathetic', 'weak', 'worthless'
}
# Observer-specific behavioral signals
WITHDRAWAL_SIGNALS = {
    'doesn\'t come out', 'stopped going', 'hasn\'t been', 'no longer',
    'used to', 'doesn\'t talk', 'avoiding', 'isolating', 'withdrawn',
    'stopped eating', 'not eating', 'skipping meals', 'hasn\'t eaten',
    'stopped sleeping', 'not sleeping', 'up all night', 'sleeping all day',
    'lost interest', 'doesn\'t care', 'given up', 'stopped laughing',
    'not themselves', 'changed', 'different person', 'barely speaks'
}

FUNCTIONING_SIGNALS = {
    'still working', 'going to work', 'managing', 'getting through',
    'holding it together', 'functioning', 'keeping up', 'showing up',
    'fine at work', 'doing okay', 'seems okay', 'laughed', 'smiled'
}

def count_signal_words(text, word_set):
    text_lower = str(text).lower()
    return sum(1 for word in word_set if word in text_lower)

def get_psychological_features(text):
    text = str(text)
    words = text.lower().split()
    sentences = sent_tokenize(text)
    
    # Signal word counts
    isolation_score = count_signal_words(text, ISOLATION_WORDS)
    hopelessness_score = count_signal_words(text, HOPELESSNESS_WORDS)
    burnout_score = count_signal_words(text, BURNOUT_WORDS)
    self_blame_score = count_signal_words(text, SELF_BLAME_WORDS)
    
    # First person singular usage (self-focus indicator)
    first_person = sum(1 for w in words if w in {'i', 'me', 'my', 'myself', 'mine'})
    first_person_ratio = first_person / max(len(words), 1)
    
    # Question count (uncertainty, seeking help signal)
    question_count = text.count('?')
    
    # Negation count
    negations = sum(1 for w in words if w in {
        'not', 'no', 'never', 'nothing', 'nobody', 
        'nowhere', 'neither', 'cant', 'wont', 'dont',
        'doesnt', 'didnt', 'isnt', 'arent', 'wasnt'
    })
    negation_ratio = negations / max(len(words), 1)
    
    # Text statistics
    word_count = len(words)
    avg_sentence_length = word_count / max(len(sentences), 1)
    
    # Exclamation ratio (emotional intensity)
    exclamation_ratio = text.count('!') / max(len(sentences), 1)

    withdrawal_score = count_signals(WITHDRAWAL_SIGNALS)
    functioning_score = count_signals(FUNCTIONING_SIGNALS)
    
    return {
        'sentiment_neg': scores['neg'],
        'sentiment_neu': scores['neu'],
        'sentiment_pos': scores['pos'],
        'sentiment_compound': scores['compound'],
        'isolation_score': count_signals(ISOLATION_WORDS),
        'hopelessness_score': count_signals(HOPELESSNESS_WORDS),
        'burnout_score': count_signals(BURNOUT_WORDS),
        'self_blame_score': count_signals(SELF_BLAME_WORDS),
        'first_person_ratio': first_person / max(word_count, 1),
        'question_count': text.count('?'),
        'negation_ratio': negations / max(word_count, 1),
        'word_count': word_count,
        'avg_sentence_length': word_count / sentence_count,
        'exclamation_ratio': text.count('!') / sentence_count,
        'withdrawal_score': withdrawal_score,
        'functioning_score': functioning_score
    }

# Test it
test_text = "I'm so exhausted. I don't know why I even bother anymore. Nobody understands what I'm going through and I feel completely alone."
print("Psychological features test:")
print(get_psychological_features(test_text))

Psychological features test:
{'isolation_score': 2, 'hopelessness_score': 0, 'burnout_score': 1, 'self_blame_score': 0, 'first_person_ratio': 0.13636363636363635, 'question_count': 0, 'negation_ratio': 0.045454545454545456, 'word_count': 22, 'avg_sentence_length': 7.333333333333333, 'exclamation_ratio': 0.0}


In [7]:
from tqdm import tqdm
tqdm.pandas()

print("Extracting sentiment features...")
sentiment_features = df['text'].progress_apply(
    lambda x: pd.Series(get_sentiment_features(x))
)

print("Extracting psychological features...")
psych_features = df['text'].progress_apply(
    lambda x: pd.Series(get_psychological_features(x))
)

# Combine everything
df_features = pd.concat([
    df[['text', 'binary_label', 'cascade_stage']],
    sentiment_features,
    psych_features
], axis=1)

print(f"\nFeature matrix shape: {df_features.shape}")
print(f"Columns: {list(df_features.columns)}")

Extracting sentiment features...


100%|██████████| 240448/240448 [03:41<00:00, 1084.66it/s]


Extracting psychological features...


100%|██████████| 240448/240448 [00:30<00:00, 7759.73it/s]


Feature matrix shape: (240448, 17)
Columns: ['text', 'binary_label', 'cascade_stage', 'sentiment_neg', 'sentiment_neu', 'sentiment_pos', 'sentiment_compound', 'isolation_score', 'hopelessness_score', 'burnout_score', 'self_blame_score', 'first_person_ratio', 'question_count', 'negation_ratio', 'word_count', 'avg_sentence_length', 'exclamation_ratio']


In [8]:
df_features.to_csv('../data/processed/kairos_features.csv', index=False)
print("Saved kairos_features.csv")
print(f"\nFeature stats by cascade stage:")
feature_cols = list(sentiment_features.columns) + list(psych_features.columns)
print(df_features.groupby('cascade_stage')[feature_cols].mean().round(3).T)

Saved kairos_features.csv

Feature stats by cascade stage:
cascade_stage         crisis  depression  stress
sentiment_neg          0.134       0.158   0.106
sentiment_neu          0.743       0.736   0.792
sentiment_pos          0.122       0.106   0.101
sentiment_compound    -0.154      -0.160  -0.044
isolation_score        0.277       0.140   0.116
hopelessness_score     0.718       0.382   0.394
burnout_score          0.062       0.045   0.038
self_blame_score       0.164       0.076   0.048
first_person_ratio     0.081       0.080   0.076
question_count         0.728       0.000   0.324
negation_ratio         0.013       0.015   0.010
word_count           131.947      74.960  85.449
avg_sentence_length   20.407      74.960  18.101
exclamation_ratio      0.092       0.000   0.022
